# Matched Saccade Behavior vs Terminal GCaMP Retina Analysis

This notebook compares two already-analyzed mouse-matched datasets:

1. **In vivo saccade behavior**
   - one row per mouse × eye × stimulus protocol × stimulus step
   - includes saccade count, frequency, total displacement, and velocity

2. **Terminal ex vivo GCaMP retina responses**
   - same mouse identity preserved
   - ~300 cells per retina
   - each cell classified as normal, attenuated, or low/non-responding
   - same stimulus protocols: contrast, frequency, brightness, spatial resolution

## Main question

Do retinal GCaMP response defects match the observed saccade defects for the same mouse and same stimulus protocol?

The shared analysis unit is:

```text
mouse_id × eye × stimulus_protocol × stimulus_value
```

The strongest first-pass method is to aggregate GCaMP cells to mouse/stimulus summaries, merge with saccade summaries, and test whether retinal defect severity predicts saccade defect severity.

## Required libraries and purpose

| Library | Purpose |
|---|---|
| `pathlib` | file/folder paths |
| `numpy` | numerical calculations |
| `pandas` | load, merge, aggregate, and save CSV tables |
| `matplotlib` | plotting |
| `seaborn` | statistical plots and heatmaps |
| `scipy.stats` | correlations and nonparametric tests |
| `statsmodels` | regression and mixed-effects models |
| `sklearn.metrics` | binary agreement/reporting |

Install:

```bash
pip install numpy pandas matplotlib seaborn scipy statsmodels scikit-learn
```

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats
import statsmodels.formula.api as smf
import statsmodels.api as sm

from sklearn.metrics import classification_report

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 60)

sns.set_context("notebook")
sns.set_style("whitegrid")

## Step 1 — Input files

Expected folder:

```text
matched_retina_behavior_project/
    data/
        saccade_summary.csv
        gcamp_cell_responses.csv
    outputs/
        tables/
        figures/
```

### `saccade_summary.csv`

Required columns:

```text
mouse_id
eye
group
stimulus_protocol
stimulus_value
saccade_count
saccade_frequency
total_eye_displacement
mean_saccade_velocity
```

### `gcamp_cell_responses.csv`

Required columns:

```text
mouse_id
eye
group
stimulus_protocol
stimulus_value
cell_id
response_class
```

`response_class` should contain:

```text
normal
attenuated
low_nonresponding
```

In [ ]:
project_dir = Path("../matched_retina_behavior_project")

data_dir = project_dir / "data"
output_dir = project_dir / "outputs"
table_dir = output_dir / "tables"
fig_dir = output_dir / "figures"

for folder in [output_dir, table_dir, fig_dir]:
    folder.mkdir(parents=True, exist_ok=True)

saccade_csv = data_dir / "saccade_summary.csv"
gcamp_csv = data_dir / "gcamp_cell_responses.csv"

print("Saccade file:", saccade_csv)
print("GCaMP file:", gcamp_csv)

## Step 2 — Load datasets

This imports the already-analyzed behavior and terminal GCaMP data.

In [ ]:
saccades = pd.read_csv(saccade_csv)
gcamp = pd.read_csv(gcamp_csv)

print("Saccade shape:", saccades.shape)
display(saccades.head())

print("GCaMP shape:", gcamp.shape)
display(gcamp.head())

## Step 3 — Validate and clean columns

Both datasets must share the same merge keys:

```text
mouse_id, eye, group, stimulus_protocol, stimulus_value
```

In [ ]:
merge_keys = ["mouse_id", "eye", "group", "stimulus_protocol", "stimulus_value"]

required_saccade_cols = merge_keys + [
    "saccade_count",
    "saccade_frequency",
    "total_eye_displacement",
    "mean_saccade_velocity"
]

required_gcamp_cols = merge_keys + [
    "cell_id",
    "response_class"
]

missing_saccade = [c for c in required_saccade_cols if c not in saccades.columns]
missing_gcamp = [c for c in required_gcamp_cols if c not in gcamp.columns]

if missing_saccade:
    raise ValueError(f"Missing saccade columns: {missing_saccade}")

if missing_gcamp:
    raise ValueError(f"Missing GCaMP columns: {missing_gcamp}")

for data in [saccades, gcamp]:
    for col in merge_keys:
        data[col] = data[col].astype(str).str.strip()

saccades["group"] = saccades["group"].str.lower()
gcamp["group"] = gcamp["group"].str.lower()

gcamp["response_class"] = (
    gcamp["response_class"]
    .astype(str)
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

for col in ["saccade_count", "saccade_frequency", "total_eye_displacement", "mean_saccade_velocity"]:
    saccades[col] = pd.to_numeric(saccades[col], errors="coerce")

print("Groups:", saccades["group"].unique())
print("Response classes:", gcamp["response_class"].unique())

## Step 4 — Aggregate GCaMP cell responses per mouse/stimulus

Cell-level GCaMP responses are converted to mouse/stimulus summaries.

Scoring:

```text
normal = 0
attenuated = 1
low/non-responding = 2
```

Then:

```text
retinal_defect_score = mean response score across cells
```

Interpretation:

```text
0 = mostly normal
1 = mostly attenuated
2 = mostly low/non-responding
```

In [ ]:
response_score_map = {
    "normal": 0,
    "attenuated": 1,
    "low_nonresponding": 2,
    "low_non_responding": 2,
    "low": 2,
    "nonresponding": 2,
    "non_responding": 2
}

gcamp["response_score"] = gcamp["response_class"].map(response_score_map)

if gcamp["response_score"].isna().any():
    print("Unmapped response classes:")
    display(gcamp[gcamp["response_score"].isna()]["response_class"].value_counts())

gcamp_summary = (
    gcamp
    .groupby(merge_keys)
    .agg(
        n_cells=("cell_id", "count"),
        retinal_defect_score=("response_score", "mean"),
        fraction_normal=("response_class", lambda x: np.mean(x == "normal")),
        fraction_attenuated=("response_class", lambda x: np.mean(x == "attenuated")),
        fraction_low_nonresponding=("response_score", lambda x: np.mean(x == 2))
    )
    .reset_index()
)

for optional_metric in ["peak_dff", "AUC"]:
    if optional_metric in gcamp.columns:
        temp = (
            gcamp
            .groupby(merge_keys)[optional_metric]
            .agg(["mean", "median"])
            .reset_index()
            .rename(columns={
                "mean": f"mean_{optional_metric}",
                "median": f"median_{optional_metric}"
            })
        )
        gcamp_summary = gcamp_summary.merge(temp, on=merge_keys, how="left")

gcamp_summary.to_csv(table_dir / "gcamp_mouse_stimulus_summary.csv", index=False)

display(gcamp_summary.head())

## Step 5 — Create control-normalized saccade defect score

For each stimulus protocol/value, controls define normal behavior.

For each behavior metric:

```text
control_z = (mouse_value - control_mean) / control_std
```

Then:

```text
saccade_defect_score = negative average control_z
```

So higher values mean worse/reduced behavior.

In [ ]:
saccade_metrics = [
    "saccade_count",
    "saccade_frequency",
    "total_eye_displacement",
    "mean_saccade_velocity"
]

control_label = "control"

def add_control_zscores(data, metric):
    out = data.copy()

    stats_df = (
        out[out["group"] == control_label]
        .groupby(["stimulus_protocol", "stimulus_value"])[metric]
        .agg(["mean", "std"])
        .reset_index()
        .rename(columns={
            "mean": f"{metric}_control_mean",
            "std": f"{metric}_control_std"
        })
    )

    out = out.merge(stats_df, on=["stimulus_protocol", "stimulus_value"], how="left")

    std_col = f"{metric}_control_std"
    mean_col = f"{metric}_control_mean"
    z_col = f"{metric}_control_z"

    out[std_col] = out[std_col].replace(0, np.nan)
    out[z_col] = (out[metric] - out[mean_col]) / out[std_col]

    return out

saccade_scored = saccades.copy()

for metric in saccade_metrics:
    saccade_scored = add_control_zscores(saccade_scored, metric)

z_cols = [f"{m}_control_z" for m in saccade_metrics]
saccade_scored["saccade_defect_score"] = -saccade_scored[z_cols].mean(axis=1)

saccade_scored.to_csv(table_dir / "saccade_mouse_stimulus_scored.csv", index=False)

display(saccade_scored.head())

## Step 6 — Merge matched retina and behavior data

Each row after merging is:

```text
mouse × eye × stimulus_protocol × stimulus_value
```

with both:

```text
retinal_defect_score
saccade_defect_score
```

In [ ]:
matched = saccade_scored.merge(
    gcamp_summary,
    on=merge_keys,
    how="inner"
)

matched.to_csv(table_dir / "matched_saccade_gcamp_mouse_stimulus_data.csv", index=False)

print("Matched shape:", matched.shape)
display(matched.head())

## Step 7 — Matched-data QC

This checks sample size after merging.

In [ ]:
qc = (
    matched
    .groupby(["group", "stimulus_protocol"])
    .agg(
        n_rows=("mouse_id", "count"),
        n_mice=("mouse_id", "nunique"),
        mean_retinal_defect=("retinal_defect_score", "mean"),
        mean_saccade_defect=("saccade_defect_score", "mean")
    )
    .reset_index()
)

qc.to_csv(table_dir / "matched_data_qc_summary.csv", index=False)

display(qc)

## Step 8 — Retinal defect vs saccade defect plot

If retinal dysfunction contributes to saccade defects, points should trend upward:

```text
more retinal defect → more saccade defect
```

In [ ]:
plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=matched,
    x="retinal_defect_score",
    y="saccade_defect_score",
    hue="group",
    style="stimulus_protocol",
    s=80
)

sns.regplot(
    data=matched,
    x="retinal_defect_score",
    y="saccade_defect_score",
    scatter=False,
    color="black"
)

plt.xlabel("Retinal GCaMP defect score")
plt.ylabel("Saccade defect score")
plt.title("Matched retinal response defect vs saccade defect")
plt.tight_layout()

out_path = fig_dir / "retinal_defect_vs_saccade_defect_scatter.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 9 — Correlation by stimulus protocol

This asks whether retina and behavior defects align within each stimulus protocol.

- Pearson correlation: linear relationship
- Spearman correlation: rank-based relationship

In [ ]:
corr_rows = []

for protocol, sub in matched.groupby("stimulus_protocol"):
    clean = sub[["retinal_defect_score", "saccade_defect_score"]].dropna()

    if clean.shape[0] >= 3:
        pearson_r, pearson_p = stats.pearsonr(
            clean["retinal_defect_score"],
            clean["saccade_defect_score"]
        )

        spearman_r, spearman_p = stats.spearmanr(
            clean["retinal_defect_score"],
            clean["saccade_defect_score"]
        )

        corr_rows.append({
            "stimulus_protocol": protocol,
            "n": clean.shape[0],
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p
        })

corr_df = pd.DataFrame(corr_rows)
corr_df.to_csv(table_dir / "retina_saccade_correlation_by_protocol.csv", index=False)

display(corr_df)

## Step 10 — Heatmaps of retinal and saccade defects by stimulus step

These plots show whether the same stimulus steps are defective in both datasets.

In [ ]:
for protocol in matched["stimulus_protocol"].unique():
    sub = matched[matched["stimulus_protocol"] == protocol].copy()

    retinal_pivot = (
        sub
        .groupby(["group", "stimulus_value"])["retinal_defect_score"]
        .mean()
        .reset_index()
        .pivot(index="group", columns="stimulus_value", values="retinal_defect_score")
    )

    saccade_pivot = (
        sub
        .groupby(["group", "stimulus_value"])["saccade_defect_score"]
        .mean()
        .reset_index()
        .pivot(index="group", columns="stimulus_value", values="saccade_defect_score")
    )

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.heatmap(retinal_pivot, annot=True, cmap="Reds", ax=axes[0])
    axes[0].set_title(f"{protocol}: retinal defect")

    sns.heatmap(saccade_pivot, annot=True, cmap="Blues", ax=axes[1])
    axes[1].set_title(f"{protocol}: saccade defect")

    plt.tight_layout()

    out_path = fig_dir / f"{protocol}_retina_saccade_defect_heatmaps.tif"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

## Step 11 — Mouse/protocol-level summaries

This collapses stimulus steps within each protocol:

```text
mouse × protocol
```

This is useful for mouse-level comparisons.

In [ ]:
protocol_summary = (
    matched
    .groupby(["mouse_id", "eye", "group", "stimulus_protocol"])
    .agg(
        retinal_protocol_defect=("retinal_defect_score", "mean"),
        saccade_protocol_defect=("saccade_defect_score", "mean"),
        mean_fraction_normal=("fraction_normal", "mean"),
        mean_fraction_attenuated=("fraction_attenuated", "mean"),
        mean_fraction_low_nonresponding=("fraction_low_nonresponding", "mean"),
        n_stimulus_steps=("stimulus_value", "nunique")
    )
    .reset_index()
)

protocol_summary.to_csv(table_dir / "mouse_protocol_defect_summary.csv", index=False)

display(protocol_summary.head())

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    data=protocol_summary,
    x="retinal_protocol_defect",
    y="saccade_protocol_defect",
    hue="group",
    style="stimulus_protocol",
    s=90
)

sns.regplot(
    data=protocol_summary,
    x="retinal_protocol_defect",
    y="saccade_protocol_defect",
    scatter=False,
    color="black"
)

plt.xlabel("Protocol-level retinal defect")
plt.ylabel("Protocol-level saccade defect")
plt.title("Mouse-matched protocol-level defect relationship")
plt.tight_layout()

out_path = fig_dir / "protocol_level_retinal_vs_saccade_defect.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 12 — Regression: do retinal defects predict saccade defects?

Model:

```text
saccade_defect_score ~ retinal_defect_score + group + stimulus_protocol
```

Interpretation:

- positive retinal coefficient: worse retinal responses predict worse saccades
- group coefficient: glaucoma effect not fully explained by retinal defect score
- protocol coefficient: stimulus protocols differ in defect severity

In [ ]:
model_df = matched.dropna(
    subset=["saccade_defect_score", "retinal_defect_score", "group", "stimulus_protocol"]
).copy()

formula = (
    "saccade_defect_score ~ "
    "retinal_defect_score + "
    "C(group) + "
    "C(stimulus_protocol)"
)

model = smf.ols(formula=formula, data=model_df).fit()

print(model.summary())

with open(table_dir / "retinal_predicts_saccade_defect_ols_summary.txt", "w") as f:
    f.write(model.summary().as_text())

## Step 13 — Interaction model

This asks whether the retina-behavior relationship differs by stimulus protocol:

```text
saccade_defect_score ~ retinal_defect_score * stimulus_protocol + group
```

In [ ]:
interaction_formula = (
    "saccade_defect_score ~ "
    "retinal_defect_score * C(stimulus_protocol) + "
    "C(group)"
)

interaction_model = smf.ols(
    formula=interaction_formula,
    data=model_df
).fit()

print(interaction_model.summary())

with open(table_dir / "retinal_saccade_protocol_interaction_model_summary.txt", "w") as f:
    f.write(interaction_model.summary().as_text())

## Step 14 — Mouse-level defect profile matching

For each mouse, compare its retinal defect profile across protocols to its saccade defect profile.

This asks:

```text
Does each mouse's pattern of retinal defects match its pattern of behavior defects?
```

In [ ]:
retina_profile = protocol_summary.pivot_table(
    index=["mouse_id", "eye", "group"],
    columns="stimulus_protocol",
    values="retinal_protocol_defect"
)

saccade_profile = protocol_summary.pivot_table(
    index=["mouse_id", "eye", "group"],
    columns="stimulus_protocol",
    values="saccade_protocol_defect"
)

common_index = retina_profile.index.intersection(saccade_profile.index)
common_cols = retina_profile.columns.intersection(saccade_profile.columns)

profile_rows = []

for idx in common_index:
    r = retina_profile.loc[idx, common_cols]
    s = saccade_profile.loc[idx, common_cols]

    valid = r.notna() & s.notna()

    if valid.sum() >= 3:
        profile_corr, profile_p = stats.spearmanr(r[valid], s[valid])
    else:
        profile_corr, profile_p = np.nan, np.nan

    profile_rows.append({
        "mouse_id": idx[0],
        "eye": idx[1],
        "group": idx[2],
        "n_protocols": int(valid.sum()),
        "retina_saccade_profile_spearman_r": profile_corr,
        "profile_p_value": profile_p
    })

profile_match = pd.DataFrame(profile_rows)
profile_match.to_csv(table_dir / "mouse_level_retina_saccade_profile_matching.csv", index=False)

display(profile_match)

In [ ]:
plt.figure(figsize=(7, 5))

sns.boxplot(
    data=profile_match,
    x="group",
    y="retina_saccade_profile_spearman_r"
)

sns.stripplot(
    data=profile_match,
    x="group",
    y="retina_saccade_profile_spearman_r",
    color="black",
    alpha=0.7
)

plt.axhline(0, linestyle="--", color="black")
plt.ylabel("Within-mouse retina/saccade profile correlation")
plt.xlabel("Group")
plt.title("Mouse-level retina-behavior profile matching")
plt.tight_layout()

out_path = fig_dir / "mouse_level_profile_match_by_group.tif"
plt.savefig(out_path, dpi=300, bbox_inches="tight")
plt.show()

## Step 15 — Binary agreement analysis

This converts continuous defect scores to binary labels:

```text
defective
not defective
```

Then it asks whether retinal defects identify the same stimulus steps with saccade defects.

In [ ]:
RETINAL_DEFECT_THRESHOLD = 0.75
SACCADE_DEFECT_THRESHOLD = 1.0

matched["retinal_defective"] = matched["retinal_defect_score"] >= RETINAL_DEFECT_THRESHOLD
matched["saccade_defective"] = matched["saccade_defect_score"] >= SACCADE_DEFECT_THRESHOLD

agreement_table = pd.crosstab(
    matched["retinal_defective"],
    matched["saccade_defective"],
    rownames=["Retinal defective"],
    colnames=["Saccade defective"]
)

agreement_table.to_csv(table_dir / "retinal_saccade_binary_defect_agreement_table.csv")

display(agreement_table)

print(classification_report(
    matched["saccade_defective"],
    matched["retinal_defective"],
    target_names=["saccade_not_defective", "saccade_defective"]
))

## Step 16 — Group/protocol summary plots

These show whether glaucoma mice show stronger retinal and behavioral defects by stimulus protocol.

In [ ]:
group_protocol_summary = (
    protocol_summary
    .groupby(["group", "stimulus_protocol"])
    .agg(
        mean_retinal_defect=("retinal_protocol_defect", "mean"),
        sem_retinal_defect=("retinal_protocol_defect", lambda x: x.std() / np.sqrt(x.count())),
        mean_saccade_defect=("saccade_protocol_defect", "mean"),
        sem_saccade_defect=("saccade_protocol_defect", lambda x: x.std() / np.sqrt(x.count())),
        n_mice=("mouse_id", "nunique")
    )
    .reset_index()
)

group_protocol_summary.to_csv(table_dir / "group_protocol_defect_summary.csv", index=False)

display(group_protocol_summary)

for y_col, title in [
    ("retinal_protocol_defect", "Protocol-level retinal defect by group"),
    ("saccade_protocol_defect", "Protocol-level saccade defect by group")
]:
    plt.figure(figsize=(9, 5))

    sns.pointplot(
        data=protocol_summary,
        x="stimulus_protocol",
        y=y_col,
        hue="group",
        errorbar="se"
    )

    plt.xticks(rotation=45, ha="right")
    plt.title(title)
    plt.tight_layout()

    out_path = fig_dir / f"{y_col}_by_group_protocol.tif"
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()

## Step 17 — Optional mixed-effects model

Because each mouse contributes multiple stimulus rows, a mixed model is preferable for final analysis:

```text
saccade_defect_score ~ retinal_defect_score * stimulus_protocol + group + mouse_effect
```

In [ ]:
try:
    mixed_model = smf.mixedlm(
        "saccade_defect_score ~ retinal_defect_score * C(stimulus_protocol) + C(group)",
        data=model_df,
        groups=model_df["mouse_id"]
    ).fit()

    print(mixed_model.summary())

    with open(table_dir / "retinal_saccade_mouse_mixed_model_summary.txt", "w") as f:
        f.write(mixed_model.summary().as_text())

except Exception as e:
    print("Mixed model could not be fit:")
    print(e)

# Final interpretation guide

## Strong evidence that GCaMP defects match saccade defects

Look for:

```text
1. glaucoma mice have higher retinal defect scores
2. glaucoma mice have higher saccade defect scores
3. retinal_defect_score positively predicts saccade_defect_score
4. correlation is strongest within matched stimulus protocols
5. each mouse's retinal defect profile matches its behavioral defect profile
```

## Example conclusion

```text
Across matched mice, retinal GCaMP response defects predicted reduced visually evoked saccades.
The relationship was strongest for contrast and spatial-resolution protocols, suggesting that
stimulus-specific retinal signaling deficits contribute to the behavioral saccade phenotype
in the glaucoma model.
```

## Important cautions

- GCaMP cells are nested within mouse, so aggregate to mouse/stimulus before matching with behavior.
- If many stimulus steps are tested, prioritize model-level inference over individual p-values.
- If both eyes are analyzed in the future, include eye as an additional repeated-measures factor.